# Phase 2: The Forecasting Showdown

In this phase, we build a comparative pipeline to predict total weekly sales. We evaluate two differing approaches:
1. **Facebook Prophet**: A generative additive model treating promotion as an external regressor.
2. **LightGBM**: A gradient-boosted tree approach built on newly engineered lag and rolling window features.

> **Goal**: Identify the most robust forecasting methodology while ensuring strictly zero temporal data leakage via `TimeSeriesSplit`.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Aggregation
We load the cleaned promotional data produced in Phase 1.

In [ ]:
data_path = '../data/processed/cleaned_data.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"COULD NOT FIND {data_path}. PLEASE RUN NOTEBOOK 01 FIRST TO EXPORT CLEANED DATA.")

df = pd.read_csv(data_path)

# Aggregate to weekly level
weekly_df = df.groupby('WEEK').agg({
    'Sales': 'sum',
    'Promo': 'mean'
}).reset_index()

base_date = pd.to_datetime('1989-09-07')
weekly_df['Date'] = base_date + pd.to_timedelta(weekly_df['WEEK'] * 7, unit='d')
weekly_df = weekly_df.sort_values('Date').reset_index(drop=True)
print(f"Aggregation complete. Observations: {len(weekly_df)}")

## 2. Feature Engineering (For LightGBM)
Creating lags and rolling windows.

In [ ]:
lgb_df = weekly_df.copy()
lgb_df['Sales_Lag1'] = lgb_df['Sales'].shift(1)
lgb_df['Sales_Lag2'] = lgb_df['Sales'].shift(2)
lgb_df['Sales_Lag4'] = lgb_df['Sales'].shift(4)
lgb_df['Sales_Rolling_Mean_4w'] = lgb_df['Sales'].shift(1).rolling(window=4).mean()
lgb_df['Sales_Rolling_Std_4w'] = lgb_df['Sales'].shift(1).rolling(window=4).std()

lgb_df = lgb_df.dropna().reset_index(drop=True)

## 3. Strict Temporal Validation Pipeline
Using TimeSeriesSplit.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
prophet_maes, prophet_rmses = [], []
lgb_maes, lgb_rmses = [], []

print("Starting CV Pipeline...")
for fold, (train_idx, test_idx) in enumerate(tscv.split(lgb_df)):
    train_data = lgb_df.iloc[train_idx]
    test_data = lgb_df.iloc[test_idx]
    
    # Prophet
    prophet_train = train_data[['Date', 'Sales', 'Promo']].rename(columns={'Date': 'ds', 'Sales': 'y'})
    prophet_test = test_data[['Date', 'Promo']].rename(columns={'Date': 'ds'})
    
    m = Prophet(weekly_seasonality=False, daily_seasonality=False)
    m.add_country_holidays(country_name='US')
    m.add_regressor('Promo')
    m.fit(prophet_train)
    
    prophet_preds = m.predict(prophet_test)['yhat'].values
    y_true = test_data['Sales'].values
    
    prophet_maes.append(mean_absolute_error(y_true, prophet_preds))
    prophet_rmses.append(root_mean_squared_error(y_true, prophet_preds))
    
    # LightGBM
    features = ['Promo', 'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag4', 'Sales_Rolling_Mean_4w', 'Sales_Rolling_Std_4w']
    X_train, y_train_lgb = train_data[features], train_data['Sales']
    X_test = test_data[features]
    
    lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=1, verbosity=-1)
    lgb_model.fit(X_train, y_train_lgb)
    lgb_preds = lgb_model.predict(X_test)
    
    lgb_maes.append(mean_absolute_error(y_true, lgb_preds))
    lgb_rmses.append(root_mean_squared_error(y_true, lgb_preds))

print("Validation Complete.")

## 4. Evaluation & Visualization
Comparing metrics and visual performance.

In [ ]:
results = pd.DataFrame({
    "Model": ["Prophet", "LightGBM"],
    "Avg MAE": [np.mean(prophet_maes), np.mean(lgb_maes)],
    "Avg RMSE": [np.mean(prophet_rmses), np.mean(lgb_rmses)]
})
display(results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
ax1.plot(test_data['Date'], y_true, label='Actual', marker='o')
ax1.plot(test_data['Date'], prophet_preds, label='Prophet', linestyle='--')
ax1.plot(test_data['Date'], lgb_preds, label='LightGBM', linestyle='-.')
ax1.set_title('Actual vs Predicted (Final Fold)')
ax1.legend()

lgb.plot_importance(lgb_model, ax=ax2, importance_type='gain')
plt.show()